# Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Actividad GA1-220501094-01-AA1-EV06**

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Contenido:

- Lección 1: carga de datos (preparación)
- Lección 2: limpieza y estandarización
- Lección 3: filtrado, ordenamiento y agrupación
- Lección 4: combinación de datasets

## Preparación del entorno

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
SALIDA.mkdir(exist_ok=True)
print(pd.__version__)

3.0.6


## Lección 1: carga de los cinco archivos

Los CSV usan `;` como separador, por eso se indica `sep=';'`.

In [ ]:
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

In [ ]:
for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

Resultado esperado:

```
orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)
```

## Lección 2: limpieza y estandarización

Orden de trabajo: duplicados, tipos, nulos, texto, duplicados implícitos y verificación. Para cada problema la decisión es **eliminar, corregir o conservar**, según lo que signifique el dato.

### 2.1 Duplicados exactos

In [ ]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

**Interpretación:** hay 30 filas (15 pedidos y sus 15 copias). Todas ocurren el miércoles a las 2 a. m., lo que apunta a un fallo del sistema y no a pedidos reales. **Decisión: eliminar.**

In [ ]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

### 2.2 Tipos de dato con `astype`

In [ ]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

**Interpretación:** al usar el tipo entero más pequeño que alcanza para cada columna, `orders` baja de 23,0 MB a 7,2 MB. Las columnas con nulos usan `Int16` (con mayúscula), que admite enteros y nulos a la vez.

### 2.3 Nulos: comprobar la causa antes de decidir

In [ ]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

**Interpretación:** cada nulo de `days_since_prior_order` es un primer pedido: no existe un pedido anterior. **Decisión: conservar el nulo.** Rellenar con 0 inventaría que el cliente volvió el mismo día.

In [ ]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

**Interpretación:** los 1 258 productos sin nombre pertenecen todos al pasillo 100 y al departamento 21 (`missing`). El producto existe, así que no se borra. **Decisión: corregir** con el valor explícito `'Unknown'`.

| Columna | Causa | Decisión |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 | Conservar el nulo |

### 2.4 Texto con los métodos `.str`

In [ ]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

**Interpretación:** había 403 nombres con espacios dobles y 16 con espacio invisible (`\xa0`). Tras la limpieza quedan 0 de cada uno, y los nombres quedan en formato título.

### 2.5 Duplicados implícitos

In [ ]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

**Interpretación:** quedan 141 nombres repetidos con `product_id` distintos, y casi todos aparecen en pedidos. **Decisión: conservar y documentar.** Borrarlos dejaría pedidos apuntando a productos inexistentes; en los análisis se agrupa por `product_id`, no por nombre.

### 2.6 Limpieza reutilizable con funciones

In [ ]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

In [ ]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Verificación superada')

### Laboratorio guiado 2: pipeline de limpieza con informe de cambios

**Encargo:** construir una secuencia de pasos de limpieza encadenados que parta del archivo crudo, se ejecute con una sola instrucción y deje un informe de lo que cambió en cada paso.

**Paso 1: recargar los datos crudos**

In [ ]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

**Paso 2: función que anota el estado de la tabla**

In [ ]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

**Paso 3: un paso de limpieza por función**

In [ ]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

**Paso 4: encadenar los pasos con `pipe`**

In [ ]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

**Paso 5: informe de cambios**

In [ ]:
informe = pd.DataFrame(registro)
informe

**Interpretación del informe:**

- Quitar duplicados eliminó 15 filas y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- Los 836 nulos de `order_prods` siguen ahí a propósito: se decidió conservarlos (ver Reto 2, punto 3).

**Paso 6: validar el resultado**

In [ ]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

**Conclusiones del laboratorio:** el pipeline produce exactamente las mismas tablas que la limpieza manual (`True True`), deja un informe con cifras de cada paso y termina con una validación automática de las reglas de calidad.

### Reto 2

**Ejercicio 1.** Duplicados exactos en `order_prods`, pares `order_id` + `product_id` repetidos y valores de `reordered`.

In [ ]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))

**Respuesta:** no hay filas duplicadas, no se repite ningún par `order_id` + `product_id` y `reordered` solo contiene 0 y 1. `order_prods` no necesita limpieza de duplicados.

**Ejercicio 2.** Nombres duplicados o con espacios en `aisles` y `departments`.

In [ ]:
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    s = df[col]
    print(nombre,
          '| duplicados:', s.duplicated().sum(),
          '| espacios al inicio o final:', (s != s.str.strip()).sum(),
          '| espacios dobles:', s.str.contains('  ', regex=False).sum(),
          '| con mayúsculas:', (s != s.str.lower()).sum())

**Respuesta:** ninguna de las dos tablas tiene nombres duplicados, espacios sobrantes ni mayúsculas mezcladas. **No necesitan limpieza de texto.**

**Ejercicio 3.** Los 836 nulos de `add_to_cart_order`.

In [ ]:
nulos_pos = order_prods[order_prods['add_to_cart_order'].isna()]
tam_pedido = order_prods.groupby('order_id').size()
ids = nulos_pos['order_id'].unique()

print('Nulos:', len(nulos_pos))
print('Pedidos con nulos:', len(ids))
print('Productos por pedido (mínimo y máximo):', tam_pedido[ids].min(), tam_pedido[ids].max())
print('Valor máximo de add_to_cart_order:', order_prods['add_to_cart_order'].max())
print('Pedidos con más de 64 productos:', (tam_pedido > 64).sum())

**Respuesta:** los 836 nulos están en 70 pedidos, y cada uno de esos pedidos tiene como mínimo 65 productos (el máximo es 127). El valor más alto que aparece en `add_to_cart_order` es 64, y los 70 pedidos con más de 64 productos son exactamente los que tienen nulos. **Explicación:** el sistema solo numera hasta la posición 64; los productos que pasan de ese límite quedan sin posición registrada. **Decisión: conservar el nulo.** La posición real es desconocida y rellenarla inventaría un dato.

**Ejercicio 4.** Nombres que cambiaron con la estandarización.

In [ ]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
con_nombre = products_crudo['product_name'].notna()
cambiaron = (products_crudo.loc[con_nombre, 'product_name'] != products.loc[con_nombre, 'product_name']).sum()
print('Nombres no nulos que cambiaron:', cambiaron)

**Respuesta:** cambiaron 6 321 nombres: 5 902 solo por el cambio a formato título y 419 por espacios dobles o invisibles (403 + 16).

**Ejercicio 5.** Nombre del día como categoría ordenada y comparación de memoria.

In [ ]:
dias = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles',
        4: 'jueves', 5: 'viernes', 6: 'sábado'}
dia_texto = orders['order_dow'].map(dias)
dia_cat = pd.Series(pd.Categorical(dia_texto, categories=list(dias.values()), ordered=True))

print('Texto:    ', round(dia_texto.memory_usage(deep=True) / 1e6, 2), 'MB')
print('Categoría:', round(dia_cat.memory_usage(deep=True) / 1e6, 2), 'MB')
print(dia_cat.min(), '<', dia_cat.max())

**Respuesta:** como texto ocupa 7,14 MB y como categoría 0,48 MB, unas 15 veces menos, porque la categoría guarda cada nombre una sola vez y usa códigos pequeños. Además, al ser ordenada permite comparar y ordenar los días de domingo a sábado.

**Ejercicio 6.** Filas de `order_prods` de productos `'Unknown'`.

In [ ]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]
print('Filas:', len(filas_unknown))
print('Pedidos distintos:', filas_unknown['order_id'].nunique())
print('Porcentaje de las filas:', round(len(filas_unknown) / len(order_prods) * 100, 2), '%')

**Respuesta:** hay 9 547 filas (0,21 % del total) en 8 254 pedidos. **No los eliminaría:** son compras reales, así que borrarlos quitaría productos de pedidos legítimos y alteraría el tamaño de esos carritos y las tasas de recompra. Se conservan marcados como `'Unknown'` y se pueden excluir solo en los análisis donde importe el nombre.

**Ejercicio 7.** `'missing'` como nulo real en una copia de `departments`.

In [ ]:
departments_copia = departments.copy()
departments_copia['department'] = departments_copia['department'].replace('missing', pd.NA)
print(departments_copia.tail(2))
print('Nulos en la copia:', departments_copia['department'].isna().sum())
print('Nulos en la original:', departments['department'].isna().sum())

**Respuesta:** en la copia, el departamento 21 queda como nulo real y la original no cambia. No conviene hacerlo en la original antes de agrupar porque `groupby` descarta por defecto las filas con clave nula: los productos del departamento `missing` desaparecerían de los conteos sin avisar.

### Documentación de resultados de limpieza (Lección 2)

| Tabla | Problema | Evidencia | Decisión |
| --- | --- | --- | --- |
| `orders` | Filas duplicadas | 15 pedidos repetidos (miércoles, 2 a. m.) | Eliminar |
| `orders` | Tipos sobredimensionados | 23,0 MB | Reducir a 7,2 MB con `astype` |
| `orders` | Nulos en `days_since_prior_order` | 28 817 (todos son primer pedido) | Conservar |
| `products` | Nulos en `product_name` | 1 258 | Rellenar con `'Unknown'` |
| `products` | Espacios dobles e invisibles | 403 y 16 | Estandarizar texto |
| `products` | Nombres repetidos con `product_id` distinto | 141 | Conservar y agrupar por `product_id` |
| `order_prods` | Nulos en `add_to_cart_order` | 836 en 70 pedidos de más de 64 productos | Conservar |
| `order_prods` | Tipos sobredimensionados | 145,4 MB | Reducir a 54,5 MB |
| `aisles`, `departments` | Sin problemas de texto ni duplicados | 0 | Sin cambios |

## Lección 3: filtrado, ordenamiento y agrupación

Antes de escribir código, cada pregunta se traduce a la frase "*métrica* por *grupo*". Por ejemplo, "¿a qué hora se compra más?" se convierte en "número de pedidos por hora".

### 3.1 Filtrado con máscaras booleanas

In [ ]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))

In [ ]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))

**Interpretación:** hay 28 817 primeros pedidos, el mismo número de nulos de `days_since_prior_order`. Combinar máscaras con `&` permite contar los pedidos de fin de semana por la mañana.

### 3.2 Filtrado con `query`

In [ ]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))

**Interpretación:** hay 1 449 pedidos hechos desde las 10 p. m. por clientes que tardaron 30 días en volver.

### 3.3 Ordenamiento con `sort_values` y `nlargest`

In [ ]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))

**Interpretación:** tres clientes llegaron al pedido número 100. En el historial ordenado, el cliente 2 aparece primero, con sus pedidos 1 y 4.

### 3.4 Agrupación con `groupby`

In [ ]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())

**Interpretación:** la hora pico es las 10 a. m., con 40 578 pedidos.

In [ ]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

**Interpretación:** domingo (84 090) y lunes (82 185) concentran más pedidos; jueves es el día con menos (59 810). La espera media entre pedidos se mueve en un rango corto, de 10,5 a 11,8 días.

### 3.5 Tabla dinámica con `pivot_table`

In [ ]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]

**Interpretación:** la tabla tiene 24 horas por 7 días. La celda más alta de las mostradas es el lunes a las 10 h, con 7 881 pedidos.

### 3.6 Valores por fila con `transform`

In [ ]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')
print(orders.shape)

**Interpretación:** 30 527 clientes tienen 5 pedidos o más, y entre todos suman 231 946 pedidos. Se elimina la columna auxiliar para dejar `orders` con sus 6 columnas.

### Laboratorio guiado 3: perfil de hábitos de compra

**Encargo:** mercadeo quiere campañas distintas según qué tan seguido compra cada cliente y necesita saber **a qué hora compra cada tipo de cliente**.

**Paso 1: pedidos por cliente**

In [ ]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()

**Interpretación:** hay 157 437 clientes; la mitad tiene 2 pedidos o menos y el máximo es 28. Los segmentos se definen con los pedidos que aparecen en este conjunto de datos.

**Paso 2: segmentar clientes con `pd.cut`** (ocasional: 1 pedido, regular: 2 a 4, frecuente: 5 o más)

In [ ]:
segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))

**Interpretación:** el 35,2 % de los clientes son ocasionales (55 357), el 45,4 % regulares (71 553) y el 19,4 % frecuentes (30 527).

**Paso 3: llevar el segmento a cada pedido con `map`**

In [ ]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

**Paso 4: resumen por segmento**

In [ ]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento

**Interpretación:** el 19,4 % de los clientes (los frecuentes) hace 231 946 de los 478 952 pedidos (48,4 %) y vuelve con una mediana de 6 días. Los ocasionales tienen una mediana de 16 días.

**Paso 5: franjas horarias y tabla cruzada**

In [ ]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil

**Interpretación:** todos los segmentos compran sobre todo en la tarde, pero los frecuentes compran más en la mañana (35,2 %) que los ocasionales (30,0 %). Esto sugiere enviar las campañas de fidelización por la mañana.

**Paso 6: combinaciones de día y franja más activas**

In [ ]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))

**Interpretación:** las combinaciones más activas son domingo en la tarde (42 340 pedidos), lunes en la tarde (37 432) y martes en la tarde (30 751).

**Paso 7: dejar `orders` como estaba**

In [ ]:
orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)

### Reto 3

**Ejercicio 1.** *Pregunta:* pedidos por hora, solo los domingos (`order_dow == 0`). Las tres horas con más pedidos.

In [ ]:
domingos = orders[orders['order_dow'] == 0]
domingos.groupby('order_hour_of_day')['order_id'].count().nlargest(3)

**Respuesta:** los domingos las horas con más pedidos son las 15 h (7 585), las 14 h (7 533) y las 13 h (7 483): el pico es en la tarde.

**Ejercicio 2.** *Pregunta:* espera media (`days_since_prior_order`) por hora. ¿Dónde es mayor y dónde menor?

In [ ]:
espera_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean()
print('Mayor:', espera_hora.idxmax(), 'h ->', round(float(espera_hora.max()), 2), 'días')
print('Menor:', espera_hora.idxmin(), 'h ->', round(float(espera_hora.min()), 2), 'días')

**Respuesta:** la espera media es mayor a las 0 h (12,52 días) y menor a las 22 h (10,62 días), una diferencia de casi 2 días.

**Ejercicio 3.** *Pregunta:* clientes con 10 pedidos o más (pedidos por cliente, con condición).

In [ ]:
pedidos_cliente = orders.groupby('user_id')['order_id'].count()
print('Clientes con 10 pedidos o más:', (pedidos_cliente >= 10).sum())
print('Máximo de pedidos de un cliente:', pedidos_cliente.max())

**Respuesta:** 6 327 clientes tienen 10 pedidos o más; el máximo es 28.

**Ejercicio 4.** *Condición:* `days_since_prior_order == 7`. Cantidad y porcentaje del total.

In [ ]:
a_los_7 = orders['days_since_prior_order'] == 7
print('Pedidos a los 7 días:', a_los_7.sum())
print('% del total de pedidos:', round(a_los_7.sum() / len(orders) * 100, 2))
print('% de los pedidos con valor de espera:',
      round(a_los_7.sum() / orders['days_since_prior_order'].notna().sum() * 100, 2))

**Respuesta:** 44 577 pedidos se hicieron 7 días después del anterior: 9,31 % del total de pedidos, o 9,9 % si solo se cuentan los que tienen valor de espera (los primeros pedidos no tienen).

**Ejercicio 5.** *Pregunta:* productos por pedido, en `order_prods`. Pedidos con 20 productos o más y pedido más grande.

In [ ]:
productos_por_pedido = order_prods.groupby('order_id').size()
print('Pedidos con 20 productos o más:', (productos_por_pedido >= 20).sum())
print('Pedido más grande:', productos_por_pedido.idxmax(), 'con', productos_por_pedido.max(), 'productos')

**Respuesta:** 49 420 pedidos tienen 20 productos o más. El pedido más grande es el 61355, con 127 productos.

**Ejercicio 6.** *Pregunta:* tasa de recompra (media de `reordered`) por posición del carrito, posiciones 1 a 5.

In [ ]:
primeras = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
primeras.groupby('add_to_cart_order')['reordered'].mean().round(3)

**Respuesta:** la tasa de recompra baja con la posición, de 0,679 en la posición 1 a 0,617 en la 5. Los primeros productos que los clientes agregan al carrito son los que más repiten.

**Ejercicio 7.** *Pregunta:* mediana de `days_since_prior_order` por tipo de día (fin de semana o resto).

In [ ]:
orders['es_fin_de_semana'] = orders['order_dow'].isin([0, 6])
print(orders.groupby('es_fin_de_semana')['days_since_prior_order'].median())
orders = orders.drop(columns='es_fin_de_semana')

**Respuesta:** la mediana de espera es de 8 días en pedidos de fin de semana y de 7 días el resto de la semana.

## Lección 4: combinación de datasets

*(pendiente)*